In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# Video Trajectory Blind Validation V1: six saved-state tail points

This notebook reuses the environment/setup/worker approach of the successfully
executed 20260928 structured-feedback notebook. It loads the original saved
states and complete UniPC histories for the same two sources at T46/47/48.
Each point is independent with spent=0. The only optimizer revision is the
continuous certified solver; directions, budgets, scales and acceptance guards
are unchanged. All six outcomes survive setup, missing-artifact and worker failures.

Publication status: immutable candidate SHA bound.

After publication, run all once in Colab. The user executes GPU work. Rebuilding
the basis must match the original full raw/masked hashes. This stage reports
real-tail acceptance only, with no MP4, sequential control or blind-detection
success claim. Results are in the new output's `six_point_run/result.json`.


In [ ]:
from pathlib import Path
import datetime, json, sys

SOURCE_SHA = '39b312669ff45de08617cae439f4d07a7ca6c37d'
CASE_IDS = ('eval_copperkettle_s2501', 'eval_paperplane_s2502')
ARMS = ('FIXED_T46', 'FIXED_T47', 'FIXED_T48')
FIXED_DENOMINATOR = {'sources': 2, 'independent_points_per_source': 3, 'real_tail_points': 6, 'mp4_slots': 0}
OUTPUT_PARENT = Path('/content/drive/MyDrive/Video-WM/Video-Trajectory-Blind-Validation-V1')
OUTPUT_PARENT.mkdir(parents=True, exist_ok=True)
stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT = OUTPUT_PARENT / stamp
OUTPUT.mkdir(exist_ok=False)
SETUP_SLOTS_PATH = OUTPUT / 'setup_slots.json'
SETUP_RECEIPT_PATH = OUTPUT / 'setup_receipt.json'
setup_slots = {
    'status': 'GLOBAL_SETUP_NOT_COMPLETED',
    'source_sha': SOURCE_SHA,
    'fixed_denominator': FIXED_DENOMINATOR,
    'cases': {
        case_id: {
            'status': 'NOT_RUN_GLOBAL_SETUP_FAILURE',
            'slots': {
                arm: {
                    'status': 'NOT_RUN_GLOBAL_SETUP_FAILURE',
                    'reason': 'SETUP_NOT_COMPLETED',
                    'path': None,
                    'score': None,
                    'decision': None,
                }
                for arm in ARMS
            },
        }
        for case_id in CASE_IDS
    },
}
SETUP_SLOTS_PATH.write_text(json.dumps(setup_slots, indent=2) + '\n', encoding='utf-8')
SETUP_RECEIPT_PATH.write_text(json.dumps({
    'status': 'SETUP_STARTED',
    'source_sha': SOURCE_SHA,
    'output_dir': str(OUTPUT),
    'python': sys.version,
    'executable': sys.executable,
}, indent=2) + '\n', encoding='utf-8')
print('fresh output:', OUTPUT, flush=True)

SAVED_RUN = Path('/content/drive/MyDrive/Video-WM/RGB-DCT-Structured-Terminal-Feedback-V1/20260928T053626644222Z')
RUN_OUTPUT = OUTPUT / 'six_point_run'
if SOURCE_SHA is None:
    raise RuntimeError('Unpublished candidate draft: bind the published source SHA with the notebook builder before Run all.')


In [ ]:
import importlib.metadata
import os
import subprocess

SETUP_LOG = OUTPUT / 'setup.log'

def write_json(path, value):
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_text(json.dumps(value, indent=2) + '\n', encoding='utf-8')
    os.replace(temporary, path)


def mark_setup_failure(reason, *, command=None, returncode=None):
    retained = json.loads(SETUP_SLOTS_PATH.read_text(encoding='utf-8'))
    retained['status'] = 'GLOBAL_SETUP_FAILED'
    retained['reason'] = reason
    for case in retained['cases'].values():
        case['status'] = 'NOT_RUN_GLOBAL_SETUP_FAILURE'
        for slot in case['slots'].values():
            slot['status'] = 'NOT_RUN_GLOBAL_SETUP_FAILURE'
            slot['reason'] = reason
    write_json(SETUP_SLOTS_PATH, retained)
    failure = {'status': 'SETUP_FAILED', 'reason': reason, 'command': command, 'returncode': returncode}
    write_json(OUTPUT / 'setup_failure.json', failure)
    receipt = json.loads(SETUP_RECEIPT_PATH.read_text(encoding='utf-8'))
    receipt.update(failure)
    write_json(SETUP_RECEIPT_PATH, receipt)


def logged_run(command, *, cwd=None, env=None, check=True, log_path=SETUP_LOG):
    with log_path.open('a', encoding='utf-8') as log:
        line = 'COMMAND ' + repr(command) + '\n'
        print(line, end='', flush=True)
        log.write(line)
        log.flush()
        child = subprocess.Popen(
            command, cwd=cwd, env=env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        )
        for line in child.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
        returncode = child.wait()
        log.write('EXIT ' + str(returncode) + '\n')
        log.flush()
    if check and returncode:
        reason = 'SETUP_COMMAND_EXIT_' + str(returncode)
        mark_setup_failure(reason, command=command, returncode=returncode)
        raise subprocess.CalledProcessError(returncode, command)
    return subprocess.CompletedProcess(command, returncode)


def version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None

print('Python:', sys.version, flush=True)
print('Executable:', sys.executable, flush=True)
logged_run([sys.executable, '-m', 'pip', '--version'])
logged_run(['apt-get', 'update', '-qq'])
logged_run(['apt-get', 'install', '-y', '-qq', 'ffmpeg'])
if version('torch') is None:
    logged_run([
        sys.executable, '-m', 'pip', 'install',
        'torch==2.11.0', 'torchvision',
        '--index-url', 'https://download.pytorch.org/whl/cu128',
    ])
logged_run([
    sys.executable, '-m', 'pip', 'install',
    'diffusers==0.40.0', 'transformers', 'accelerate', 'ftfy',
    'sentencepiece', 'safetensors', 'huggingface_hub', 'numpy', 'Pillow',
])


In [ ]:
environment_check_code = r'''import importlib.metadata
import json
import os
import shutil
import sys
import traceback
from pathlib import Path

receipt_path = Path(os.environ['RGB_DCT_SETUP_ENVIRONMENT_RECEIPT'])
packages = (
    'torch', 'torchvision', 'diffusers', 'transformers', 'accelerate',
    'ftfy', 'sentencepiece', 'safetensors', 'huggingface_hub', 'numpy', 'Pillow',
)
receipt = {
    'status': 'FAILED',
    'python': sys.version,
    'executable': sys.executable,
    'packages': {},
    'ffmpeg': shutil.which('ffmpeg'),
    'ffprobe': shutil.which('ffprobe'),
    'torch_version_policy': 'record_and_api_check',
    'cuda_build_suffix_policy': 'record_only',
}
for name in packages:
    try:
        receipt['packages'][name] = importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        receipt['packages'][name] = None
try:
    import torch
    import diffusers
    from diffusers import AutoencoderKLWan, UniPCMultistepScheduler, WanPipeline

    torch_version = str(torch.__version__)
    receipt.update({
        'torch_version': torch_version,
        'torch_public_version': torch_version.split('+', 1)[0],
        'torch_build_suffix': torch_version.split('+', 1)[1] if '+' in torch_version else None,
        'torch_cuda_runtime': torch.version.cuda,
        'diffusers_version': diffusers.__version__,
        'cuda_available': torch.cuda.is_available(),
        'cuda_device_count': torch.cuda.device_count(),
        'cuda_device_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        'cuda_device_capability': list(torch.cuda.get_device_capability(0)) if torch.cuda.is_available() else None,
        'cuda_bfloat16_supported': bool(torch.cuda.is_bf16_supported()) if torch.cuda.is_available() else False,
        'wan_pipeline_import': WanPipeline is not None,
        'wan_vae_import': AutoencoderKLWan is not None,
    })
    assert receipt['ffmpeg'] and receipt['ffprobe'], 'ffmpeg and ffprobe are required'
    assert diffusers.__version__ == '0.40.0', diffusers.__version__
    assert receipt['cuda_available'], 'CUDA torch required'

    scheduler = UniPCMultistepScheduler(
        prediction_type='flow_prediction', thresholding=False,
        predict_x0=True, lower_order_final=True, use_flow_sigmas=True,
        flow_shift=3.0, final_sigmas_type='zero',
    )
    scheduler.set_timesteps(4, device='cuda')
    sample = torch.zeros((1, 1, 2, 2), device='cuda', dtype=torch.float32)
    model_output = torch.ones_like(sample)
    previous = scheduler.step(model_output, scheduler.timesteps[0], sample, return_dict=False)[0]
    assert scheduler.step_index == 1 and bool(torch.isfinite(previous).all()), 'native UniPC forward smoke failed'
    receipt['native_unipc_forward_smoke'] = True
    receipt['status'] = 'VALID'
except Exception as exc:
    receipt['reason'] = f'{type(exc).__name__}: {exc}'
    receipt['traceback'] = traceback.format_exc()
finally:
    receipt_path.write_text(json.dumps(receipt, indent=2) + '\n', encoding='utf-8')
if receipt['status'] != 'VALID':
    raise RuntimeError(receipt['reason'])
'''
setup_environment_path = OUTPUT / 'setup_environment_receipt.json'
check_env = os.environ.copy()
check_env['RGB_DCT_SETUP_ENVIRONMENT_RECEIPT'] = str(setup_environment_path)
completed = logged_run(
    [sys.executable, '-u', '-c', environment_check_code],
    env=check_env,
    check=False,
)
if completed.returncode:
    if setup_environment_path.exists():
        setup_environment = json.loads(setup_environment_path.read_text(encoding='utf-8'))
        reason = 'ENVIRONMENT_CHECK:' + setup_environment.get('reason', 'fresh child failed')
    else:
        reason = 'ENVIRONMENT_CHECK:fresh child produced no receipt'
    mark_setup_failure(reason, command=[sys.executable, '-u', '-c', '<environment_check_code>'], returncode=completed.returncode)
    raise subprocess.CalledProcessError(completed.returncode, [sys.executable, '-u', '-c', '<environment_check_code>'])
print('fresh-child environment receipt:', setup_environment_path, flush=True)


In [ ]:
REPO = Path('/content/SC-SSTW-VIDEO-TRAJECTORY-BLIND-VALIDATION-' + stamp)
try:
    logged_run(['git', 'clone', '--filter=blob:none', 'https://github.com/RICHAAARC/SC-SSTW.git', str(REPO)])
    logged_run(['git', '-C', str(REPO), 'fetch', '--depth', '2', 'origin', 'dev/video-trajectory-blind-validation-v1'])
    logged_run(['git', '-C', str(REPO), 'checkout', '--detach', SOURCE_SHA])
    actual = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
    attached = subprocess.run(
        ['git', '-C', str(REPO), 'symbolic-ref', '-q', 'HEAD'],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True,
    ).returncode == 0
    porcelain = subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain'], text=True)
    if actual != SOURCE_SHA:
        raise RuntimeError('immutable source SHA readback mismatch')
    if attached:
        raise RuntimeError('source checkout is not detached')
    if porcelain:
        raise RuntimeError('source checkout is not clean')
    source_files = {
        'protocol': REPO / 'docs/video_trajectory_blind_validation_v1.md',
        'config': REPO / 'experiments/wan_state_clock/configs/video_trajectory_blind_validation_v1.json',
        'runner': REPO / 'experiments/wan_state_clock/video_trajectory_blind_validation_run.py',
        'method': REPO / 'main/tube_state/rgb_dct_continuous_solver.py',
        'backend': REPO / 'runtime/wan/rgb_dct_structured_feedback_backend.py',
    }
    source_files['shared_real_loop'] = REPO / 'experiments/wan_state_clock/rgb_dct_structured_terminal_feedback_run.py'
    import hashlib
    source_receipt = {
        'status': 'VALID',
        'expected_sha': SOURCE_SHA,
        'actual_sha': actual,
        'detached_head': not attached,
        'clean_checkout': not bool(porcelain),
        'repo': str(REPO),
        'files': {
            name: {'path': str(path), 'sha256': hashlib.sha256(path.read_bytes()).hexdigest()}
            for name, path in source_files.items()
        },
    }
    write_json(OUTPUT / 'source_receipt.json', source_receipt)
except Exception as exc:
    reason = f'SOURCE_CHECKOUT:{type(exc).__name__}: {exc}'
    mark_setup_failure(reason)
    raise
print('detached source commit:', actual, flush=True)


In [ ]:
setup_environment = json.loads(setup_environment_path.read_text(encoding='utf-8'))
if setup_environment.get('status') != 'VALID':
    reason = 'ENVIRONMENT_CHECK:' + setup_environment.get('reason', 'invalid receipt')
    mark_setup_failure(reason)
    raise RuntimeError(reason)
retained = json.loads(SETUP_SLOTS_PATH.read_text(encoding='utf-8'))
retained['status'] = 'SETUP_COMPLETE_AWAITING_RESULT'
retained['reason'] = None
for case in retained['cases'].values():
    case['status'] = 'PENDING_RUNNER_RESULT'
    for slot in case['slots'].values():
        slot['status'] = 'PENDING_RUNNER_RESULT'
        slot['reason'] = None
write_json(SETUP_SLOTS_PATH, retained)
write_json(SETUP_RECEIPT_PATH, {
    'status': 'SETUP_COMPLETE',
    'source_sha': SOURCE_SHA,
    'output_dir': str(OUTPUT),
    'repo': str(REPO),
    'source_receipt': str(OUTPUT / 'source_receipt.json'),
    'environment_receipt': str(setup_environment_path),
    'python': sys.version,
    'executable': sys.executable,
})
print('setup complete; fixed runner will now create result.json before workers', flush=True)


In [ ]:
RUN_LOG = OUTPUT / 'run.log'
run_env = os.environ.copy()
run_env['PYTHONUNBUFFERED'] = '1'
command = [
    sys.executable, '-u', '-m',
    'experiments.wan_state_clock.video_trajectory_blind_validation_run',
    '--saved-run', str(SAVED_RUN),
    '--output', str(RUN_OUTPUT), '--execute-real-tail',
]
completed = logged_run(command, cwd=REPO, env=run_env, check=False, log_path=RUN_LOG)
RESULT_PATH = RUN_OUTPUT / 'result.json'
write_json(OUTPUT / 'execution_receipt.json', {
    'command': command,
    'cwd': str(REPO),
    'returncode': completed.returncode,
    'result_path': str(RESULT_PATH),
    'result_exists': RESULT_PATH.is_file(),
})
retained = json.loads(SETUP_SLOTS_PATH.read_text(encoding='utf-8'))
if not RESULT_PATH.is_file():
    retained['status'] = 'RUNNER_FAILED_NO_RESULT'
    retained['reason'] = 'RUNNER_DID_NOT_PRODUCE_RESULT'
    for case in retained['cases'].values():
        case['status'] = 'NOT_RUN_RUNNER_FAILURE'
        for slot in case['slots'].values():
            slot['status'] = 'NOT_RUN_RUNNER_FAILURE'
            slot['reason'] = 'RUNNER_DID_NOT_PRODUCE_RESULT'
    write_json(SETUP_SLOTS_PATH, retained)
    raise FileNotFoundError('runner produced no retained result.json; inspect setup_slots.json and run.log')
retained['status'] = 'SUPERSEDED_BY_RESULT_JSON'
retained['result_path'] = str(RESULT_PATH)
write_json(SETUP_SLOTS_PATH, retained)
print('runner return code:', completed.returncode, flush=True)
print('retained result:', RESULT_PATH, flush=True)


In [ ]:
result = json.loads(RESULT_PATH.read_text(encoding='utf-8'))
if result.get('source_receipt', {}).get('git_head') != SOURCE_SHA:
    raise RuntimeError('result source SHA mismatch')
if result.get('fixed_denominator') != FIXED_DENOMINATOR:
    raise RuntimeError('fixed six-point denominator mismatch')
if tuple(result.get('cases', {}).keys()) != CASE_IDS:
    raise RuntimeError('fixed source roster mismatch')
print('status:', result.get('status'))
print('fixed point counts:', result.get('point_counts'))
for case_id in CASE_IDS:
    case = result['cases'][case_id]
    if tuple(case['controls']) != ARMS:
        raise RuntimeError('fixed point roster mismatch')
    print(case_id, case.get('status'), case.get('stage'))
    for arm in ARMS:
        rows = case['controls'][arm]
        for row in rows:
            print(arm, row.get('outcome'), row.get('accepted_scale'), row.get('error'))
            for trial in row.get('candidates', []):
                print('  scale', trial['scale'], trial['decision'], trial.get('C'),
                      trial.get('lost_groups'), trial.get('true_hinge_improvement'))
print('evidence ceiling:', result['evidence_ceiling'])
print('retained result:', RESULT_PATH)
